# Two Pointers: Sliding Window and Subarray Sums

**Session:** 2026-11-02 · Recursion and Two Pointer  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

A **subarray** is defined as a contiguous non-empty sequence of elements within an array. The **Sliding Window** technique uses two pointers (`start` and `end`, or `j` and `i`) that both move in the same forward direction to inspect candidate subarrays dynamically in linear time.


## 1. Problem Specification: Subarray Sum Equals K

Given an array of integers `nums` where $a_i \ge 0$ and an integer `key`, return the total number of contiguous subarrays whose elements sum to `key`.

### Brute Force Enumeration ($O(n^2)$)
Inspect every starting index $i \in [0, n-1]$ and compute the cumulative sum for each ending index $j \in [i, n-1]$.


In [ ]:
from collections.abc import Sequence


def subarray_sum_brute_force(nums: Sequence[int], key: int) -> int:
    """Compute count of subarrays summing to key via nested loops.

    Time Complexity: O(n^2)
    Auxiliary Space: O(1)
    """
    n = len(nums)
    count = 0
    for i in range(n):
        total = 0
        for j in range(i, n):
            total += nums[j]
            if total == key:
                count += 1
    return count


nums_lecture = [1, 2, 3, 5, 6]
key_lecture = 5
print("Brute force count:", subarray_sum_brute_force(nums_lecture, key_lecture))
assert subarray_sum_brute_force(nums_lecture, key_lecture) == 2

## 2. Sliding Window / Two Pointers for Non-Negative Arrays ($O(n)$)

Because all numbers are non-negative ($a_i \ge 0$), the subarray sum possesses **monotonicity**:
- Advancing the leading pointer $i$ strictly increases or maintains the total sum.
- Advancing the trailing pointer $j$ strictly decreases or maintains the total sum.

This allows us to maintain a dynamic window $[j, i)$ in $O(n)$ total operations.


In [ ]:
def subarray_sum_two_pointer(nums: Sequence[int], key: int) -> int:
    """Two-pointer sliding window implementation from lecture slides.

    Time Complexity: O(n)
    Auxiliary Space: O(1)
    """
    i = 0
    j = 0
    count = 0
    total = 0
    n = len(nums)

    while i < n and j < n:
        if total + nums[i] < key:
            total += nums[i]
            i += 1
        elif total + nums[i] > key:
            total -= nums[j]
            j += 1
        else:
            count += 1
            total += nums[i]
            i += 1
    return count


print("Two pointers count:", subarray_sum_two_pointer(nums_lecture, key_lecture))
assert subarray_sum_two_pointer(nums_lecture, key_lecture) == 2

## 3. Step-by-Step Execution Trace (Replicating Lecture Slides 43–61)

Let `nums = [1, 2, 3, 5, 6]` and `key = 5`:

| Step | $i$ | $j$ | `total` | `total + nums[i]` | Condition | Action | `count` |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| 1 | 0 | 0 | 0 | $0 + 1 = 1 < 5$ | `< key` | `total += 1`, $i \leftarrow 1$ | 0 |
| 2 | 1 | 0 | 1 | $1 + 2 = 3 < 5$ | `< key` | `total += 2`, $i \leftarrow 2$ | 0 |
| 3 | 2 | 0 | 3 | $3 + 3 = 6 > 5$ | `> key` | `total -= nums[0] = 2`, $j \leftarrow 1$ | 0 |
| 4 | 2 | 1 | 2 | $2 + 3 = 5 == 5$ | `== key` | `count += 1`, `total += 3`, $i \leftarrow 3$ | **1** |
| 5 | 3 | 1 | 5 | $5 + 5 = 10 > 5$ | `> key` | `total -= nums[1] = 3`, $j \leftarrow 2$ | 1 |
| 6 | 3 | 2 | 3 | $3 + 5 = 8 > 5$ | `> key` | `total -= nums[2] = 0`, $j \leftarrow 3$ | 1 |
| 7 | 3 | 3 | 0 | $0 + 5 = 5 == 5$ | `== key` | `count += 1`, `total += 5`, $i \leftarrow 4$ | **2** |
| 8 | 4 | 3 | 5 | $5 + 6 = 11 > 5$ | `> key` | `total -= nums[3] = 0`, $j \leftarrow 4$ | 2 |
| 9 | 4 | 4 | 0 | $0 + 6 = 6 > 5$ | `> key` | `total -= nums[4] = -6`, $j \leftarrow 5$ | 2 |
| End | 4 | 5 | - | - | $j \ge n$ | Loop terminates | **2** |


In [ ]:
def trace_subarray_sum(nums: Sequence[int], key: int) -> None:
    """Execute step-by-step trace printing state changes."""
    i, j, count, total = 0, 0, 0, 0
    n = len(nums)
    step = 0
    print(
        f"{'Step':<5} | {'i':<3} | {'j':<3} | {'total':<6} | {'Condition':<10} | {'Action':<25} | {'Count'}"
    )
    print("-" * 70)
    while i < n and j < n:
        step += 1
        cand = total + nums[i]
        if cand < key:
            cond = "< key"
            action = f"total += {nums[i]}, i++"
            total += nums[i]
            i += 1
        elif cand > key:
            cond = "> key"
            action = f"total -= nums[{j}]({nums[j]}), j++"
            total -= nums[j]
            j += 1
        else:
            cond = "== key"
            count += 1
            action = f"count++, total += {nums[i]}, i++"
            total += nums[i]
            i += 1
        print(f"{step:<5} | {i:<3} | {j:<3} | {total:<6} | {cond:<10} | {action:<25} | {count}")


trace_subarray_sum(nums_lecture, key_lecture)

## 4. Generalization: Handling Negative Numbers with Prefix Sums

When the array contains negative numbers, monotonicity is violated. A window that expands could decrease in sum, and shrinking could increase in sum.

The general solution uses **Prefix Sums with a Hash Map**:
- Let $P[k] = \sum_{m=0}^{k-1} a_m$.
- A subarray sum from $j$ to $i$ is $P[i+1] - P[j] = \text{key} \implies P[j] = P[i+1] - \text{key}$.
- By tracking frequencies of previously seen prefix sums in a hash map, we achieve $O(n)$ time complexity for any arbitrary integer array.


In [ ]:
from collections import defaultdict


def subarray_sum_prefix_hash(nums: Sequence[int], key: int) -> int:
    """Count subarrays summing to key for arbitrary integers (negative and positive).

    Time Complexity: O(n)
    Auxiliary Space: O(n)
    """
    count = 0
    current_sum = 0
    prefix_counts: dict[int, int] = defaultdict(int)
    prefix_counts[0] = 1

    for x in nums:
        current_sum += x
        count += prefix_counts[current_sum - key]
        prefix_counts[current_sum] += 1

    return count


# Test on array with negative numbers
nums_with_neg = [1, -1, 1, 1, -1, 1]
print(
    "Subarrays summing to 2 in array with negatives:",
    subarray_sum_prefix_hash(nums_with_neg, 2),
)
assert subarray_sum_prefix_hash(nums_with_neg, 2) == 4